# GCL GPU Environment — Ordered-Path OOD Stress Lane

This notebook turns a **free Colab T4 runtime** into a GCL live-lane GPU
environment, with zero local setup:

1. clone the repo (public GitHub remote),
2. provision the pinned GPU stack (`.gcl-python-gpu` venv + `requirements/gcl-python-gpu.txt`),
3. validate the host with `scripts/gpu/gpu-smoke.py` (device, throughput, determinism, hooks),
4. run the **ordered-path OOD stress lane** live on the T4 with the recorded
   France configuration (6 OOD paraphrases, rescue + replacement search),
5. assemble the suite (`scripts/gpu/wrap-lane-suite.py`) and verify it with
   `tools/llm_reader/verify_ordered_path_ood_stress.py` (independent recomputation),
6. show the why-report section and zip all artifacts for download.

**Claim boundary:** numbers produced here describe this runtime (T4, fp32,
pinned torch build) and this lane configuration only. They are not compared
against other devices, and gate outcomes do not transfer between hosts.

Runtime → *Change runtime type* → **T4 GPU** before running.

In [ ]:
!nvidia-smi

In [ ]:
import os

REPO_URL = "https://github.com/pistakugli/genetic-code-lab.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
COMMIT = "HEAD"  #@param {type:"string"}

os.environ["HF_HOME"] = "/content/hf-cache"

!rm -rf /content/genetic-code-lab
!git clone --branch {BRANCH} {REPO_URL} /content/genetic-code-lab
%cd /content/genetic-code-lab
!git checkout {COMMIT}
!git log --oneline -1

In [ ]:
# Provision: .gcl-python-gpu venv + pinned cu128 torch stack + warm GPT-2 cache
# + gpu-smoke validation. Idempotent; safe to re-run.
!bash scripts/gpu/provision-gpu-env.sh

In [ ]:
# Live lane run on the T4: recorded France OOD-stress configuration
# (6 OOD paraphrases, layer-shift rescue, replacement path search + transfer).
# GCL_DEVICE pins the device; the lane records it in the artifact metadata.
lane_dir = "benchmarks/scratch/colab-t4-ood-stress"
!mkdir -p {lane_dir}
%cd /content/genetic-code-lab/tools
!GCL_DEVICE=cuda ../.gcl-python-gpu/bin/python llm-reader-ordered-path-stability.py \
    --adapter gpt2 --model gpt2 --local-files-only \
    --input-report ../benchmarks/release/nn-live-gpt2-ordered-residual-path-circuit-finder/gpt2-50-facts-ordered-residual-path/report.json \
    --subjects France --case-count 1 --paraphrase-count 6 --paraphrase-profile ood \
    --layer-shifts="-2,-1,1,2" --min-layer-control-count 2 --min-layer-margin 0.001 \
    --min-original-drop 0.05 --min-paraphrase-drop-ratio 0.10 --min-paraphrase-pass-rate 0.66 \
    --rescue-broken-paraphrases --rescue-layer-shifts="-3,-2,-1,1,2,3" --max-rescue-attempts 2 \
    --replacement-candidate-count 10 --replacement-max-pair-attempts 32 --replacement-max-set-attempts 24 \
    --min-layer-specificity-rate 0.8 \
    --device cuda \
    --output-json ../{lane_dir}/report.json \
    --output-md ../{lane_dir}/report.md

In [ ]:
# Assemble the release-format suite (python wrapper; no Node.js needed).
%cd /content/genetic-code-lab
!.gcl-python-gpu/bin/python scripts/gpu/wrap-lane-suite.py \
    --paraphrase-profile ood \
    --result id=gpt2,label=GPT-2,adapter=gpt2,model=gpt2,report={lane_dir}/report.json \
    --output-dir {lane_dir}-suite

In [ ]:
# Independent verification: recompute break classification, change kinds,
# transfer diagnoses, and all recorded numbers from raw artifact fields.
# A green run here means the T4 artifact is release-grade evidence.
%cd /content/genetic-code-lab/tools
!../.gcl-python-gpu/bin/python -m llm_reader.verify_ordered_path_ood_stress \
    --report ../{lane_dir}-suite/report.json

In [ ]:
# Show the why-report section of the lane markdown.
from pathlib import Path

markdown = Path(f"/content/genetic-code-lab/{lane_dir}/report.md").read_text(encoding="utf-8")
start = markdown.find("## Why the path broke")
if start == -1:
    start = markdown.find("# Why")
print(markdown[start:start + 4000] if start != -1 else markdown[:4000])

In [ ]:
# Zip all artifacts (lane report, suite, manifest, gpu-smoke) for download.
!cd /content/genetic-code-lab && zip -r /content/gcl-colab-t4-artifacts.zip \
    {lane_dir} {lane_dir}-suite benchmarks/scratch/gpu-smoke
from google.colab import files
files.download("/content/gcl-colab-t4-artifacts.zip")

## What this established

- A free Colab T4 is a **sufficient GPU environment** for the ordered-path
  OOD stress lane: provisioning, live run, suite assembly, and independent
  verification all completed on the rented runtime.
- The verifier accepted the T4 artifact (`device: cuda`), which a CPU host
  cannot produce — that rule is the release gate working as designed.
- To run the full multi-model gate (GPT-2, Pythia, Llama, Mistral) use a host
  with Node.js and `npm run benchmark:nn-live-ordered-path-ood-stress:all`
  (see `docs/gpu-environments.md`).